In [ ]:
@file:DependsOn("com.charleskorn.kaml:kaml-jvm:0.85.0")
%use ktor-client

In [ ]:
val faVersion = "7.0.1" // IMPORTANT: Keep this in sync with `libs.versions.font.awesome`
val rawIconData = http
    .get("https://raw.githubusercontent.com/FortAwesome/Font-Awesome/refs/tags/$faVersion/metadata/icon-families.yml")
    .bodyAsText()

In [ ]:
import com.charleskorn.kaml.Yaml
import com.charleskorn.kaml.yamlMap

val iconFamilies = Yaml.default.parseToYamlNode(rawIconData).yamlMap.entries

println("Loaded ${iconFamilies.size} icon families")

In [9]:
import com.charleskorn.kaml.YamlList
import com.charleskorn.kaml.YamlMap
import com.charleskorn.kaml.yamlMap
import kotlin.io.path.Path
import kotlin.io.path.appendLines
import kotlin.io.path.writeLines
import kotlin.io.path.writeText
import kotlin.io.path.writer

val iconsByStyle = iconFamilies.flatMap { (icon, data) ->
    val iconName = icon.content
    val styles = data.yamlMap
        .get<YamlMap>("familyStylesByLicense")!!
        .get<YamlList>("free")!!
        .items.map { it.yamlMap.getScalar("style")!!.content }
        // The "brand" version of the `font-awesome` and `web-awesome` icons is the same as the "solid" version,
        // so we remove them to simplify the code generation logic.
        .let { styles ->
            when (iconName) {
                "font-awesome", "web-awesome" -> {
                    check("solid" in styles)
                    styles - "brands"
                }

                else -> styles
            }
        }
    styles.map { it to iconName }
}.groupBy(keySelector = { it.first }, valueTransform = { it.second })

In [10]:
import kotlin.io.path.appendText

Path("fa-icon-list.txt").run {
    writeText("# Font Awesome $faVersion\n")
    appendText("# Generated by `FaIconsList.ipynb`. Used by the `generateIcons` Gradle task.\n")
    appendLines(iconsByStyle.map { (style, icons) -> "$style=${icons.joinToString(",")}" })
}

fa-icon-list.txt